# 03 — Baseline Model\n**ENAI 603 | WMATA Metro Delay Prediction**\n\nModels trained in this notebook:\n1. **Majority-class baseline** (always predict most frequent class)\n2. **Logistic Regression** (temporal + line + rolling features)\n3. **Random Forest** (all features)\n\nEvaluation: AUC-ROC, precision, recall, F1. Time-series aware split (train on earlier data, test on later).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    RocCurveDisplay, f1_score, precision_score, recall_score
)

sns.set_theme(style="whitegrid", font_scale=1.1)

PROJ = os.path.abspath(os.path.join(os.getcwd(), ".."))
FIG_DIR = os.path.join(PROJ, "reports", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(PROJ, "data", "features.csv"), parse_dates=["collected_at"])
print(f"Loaded {len(df):,} rows")

## 1. Prepare Features & Time-Series Split

In [ ]:
# Define feature groups
NUMERIC_FEATURES = [
    "hour", "day_of_week", "is_weekend", "is_rush_hour",
    "num_lines", "is_terminal", "lat", "lon",
    "minutes_num", "num_trains_at_station", "avg_cars_at_station",
    "delay_rate_30min", "line_delay_rate_30min",
    "active_incident", "incident_is_delay",
    "scheduled_headway_min",
]

CATEGORICAL_FEATURES = ["line"]

TARGET = "is_delayed"

# Drop rows with NaN in features we'll use
feature_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
df_model = df[feature_cols + [TARGET, "collected_at"]].copy()
df_model = df_model.dropna(subset=[TARGET])

# Fill NaN in numeric features with median
for col in NUMERIC_FEATURES:
    if df_model[col].isnull().any():
        df_model[col] = df_model[col].fillna(df_model[col].median())

# Time-series split: train on first 80% chronologically, test on last 20%
df_model = df_model.sort_values("collected_at").reset_index(drop=True)
split_idx = int(len(df_model) * 0.8)

train = df_model.iloc[:split_idx]
test = df_model.iloc[split_idx:]

X_train = train[feature_cols]
y_train = train[TARGET]
X_test = test[feature_cols]
y_test = test[TARGET]

print(f"Train: {len(train):,} rows ({train['collected_at'].min()} → {train['collected_at'].max()})")
print(f"Test:  {len(test):,} rows ({test['collected_at'].min()} → {test['collected_at'].max()})")
print(f"\nTrain delay rate: {y_train.mean():.2%}")
print(f"Test delay rate:  {y_test.mean():.2%}")

In [ ]:
# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
    ]
)

## 2. Model Training & Evaluation

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """Train, predict, and return metrics dict."""
    model.fit(X_tr, y_tr)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_te)[:, 1]
    else:
        y_prob = model.predict(X_te).astype(float)

    y_pred = model.predict(X_te)
    auc = roc_auc_score(y_te, y_prob)

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  AUC-ROC:   {auc:.4f}")
    print(f"  Precision: {precision_score(y_te, y_pred):.4f}")
    print(f"  Recall:    {recall_score(y_te, y_pred):.4f}")
    print(f"  F1:        {f1_score(y_te, y_pred):.4f}")
    print()
    print(classification_report(y_te, y_pred, target_names=["On-time", "Delayed"]))

    return {"name": name, "model": model, "auc": auc, "y_prob": y_prob, "y_pred": y_pred}


results = []

# 1. Majority-class baseline
dummy = DummyClassifier(strategy="most_frequent")
results.append(evaluate_model("Majority Baseline", dummy, X_train, y_train, X_test, y_test))

# 2. Logistic Regression
lr_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
results.append(evaluate_model("Logistic Regression", lr_pipe, X_train, y_train, X_test, y_test))

# 3. Random Forest
rf_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=20,
        class_weight="balanced", random_state=42, n_jobs=-1
    )),
])
results.append(evaluate_model("Random Forest", rf_pipe, X_train, y_train, X_test, y_test))

## 3. ROC Curves Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
colors = ["gray", "#1976D2", "#388E3C"]
for res, color in zip(results, colors):
    RocCurveDisplay.from_predictions(
        y_test, res["y_prob"],
        name=f"{res['name']} (AUC={res['auc']:.3f})",
        ax=axes[0], color=color
    )
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_title("ROC Curves")

# AUC comparison bar chart
names = [r["name"] for r in results]
aucs = [r["auc"] for r in results]
bar_colors = colors[:len(results)]
axes[1].barh(names, aucs, color=bar_colors)
axes[1].set_xlabel("AUC-ROC")
axes[1].set_title("Model Comparison")
axes[1].set_xlim(0.4, 1.0)
axes[1].axvline(0.80, color="red", ls="--", alpha=0.5, label="Target (0.80)")
axes[1].legend()
for i, v in enumerate(aucs):
    axes[1].text(v + 0.01, i, f"{v:.3f}", va="center", fontweight="bold")

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "baseline_roc_curves.png"), dpi=150)
plt.show()

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, res in zip(axes, results):
    cm = confusion_matrix(y_test, res["y_pred"])
    sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", ax=ax,
                xticklabels=["On-time", "Delayed"],
                yticklabels=["On-time", "Delayed"])
    ax.set_title(f"{res['name']}\nAUC={res['auc']:.3f}")
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "baseline_confusion_matrices.png"), dpi=150)
plt.show()

## 5. Feature Importance (Random Forest)

In [ ]:
# Extract feature names after preprocessing
rf_result = results[2]
rf_model = rf_result["model"]
ohe_features = list(rf_model.named_steps["pre"]
                    .named_transformers_["cat"]
                    .get_feature_names_out(CATEGORICAL_FEATURES))
all_feature_names = NUMERIC_FEATURES + ohe_features

importances = rf_model.named_steps["clf"].feature_importances_
feat_imp = pd.Series(importances, index=all_feature_names).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
feat_imp.tail(15).plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Feature Importance (Gini)")
ax.set_title("Top 15 Features — Random Forest")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "baseline_feature_importance.png"), dpi=150)
plt.show()

## 6. Summary Table

In [ ]:
summary = pd.DataFrame([
    {
        "Model": r["name"],
        "AUC-ROC": r["auc"],
        "Precision": precision_score(y_test, r["y_pred"]),
        "Recall": recall_score(y_test, r["y_pred"]),
        "F1": f1_score(y_test, r["y_pred"]),
    }
    for r in results
])
summary = summary.round(4)
print(summary.to_string(index=False))

## 7. Next Steps\n\n- **Hyperparameter tuning** with `TimeSeriesSplit` cross-validation\n- **XGBoost / LightGBM** for potential AUC improvement\n- **Feature selection** — drop low-importance features, address GTFS headway missingness\n- **More data** — pipeline continues collecting; re-run `build_features.py` before final submission\n- **Threshold tuning** — optimize classification threshold for best F1 / business utility